In [1]:
import os
# Enforce single GPU to prevent DataParallel / multi-GPU synchronization conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import precision_recall_fscore_support
from huggingface_hub import login, hf_hub_download
import zipfile

In [ ]:
# ==========================================
# 1. AUTHENTICATION & DATA LOADING
# ==========================================
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

# Dataset repository from your Phase 4 setup
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-modern-decoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_cjpe_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_4_CJPE/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 4 CJPE Data...")
train_data = load_cjpe_file("Train/train.json")
dev_data = load_cjpe_file("Train/dev.json")

print(f"Loaded {len(train_data)} training examples and {len(dev_data)} development examples.")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Loaded 37387 training examples and 1988 development examples.


In [3]:
DEBUG_MODE = False

if DEBUG_MODE:
    print("⚠️ DEBUG MODE ACTIVE: Subsampling dataset for rapid validation.")
    train_data = train_data[:100]
    dev_data = dev_data[:20]

In [4]:
# ==========================================
# 2. TOKENIZER & LONG-DOCUMENT SLIDING WINDOW CHUNKING
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"

def chunk_long_document(text, tokenizer, max_length=1024, stride=256):
    """
    Long-document handling technique: Sliding-window document chunking 
    to process extensive legal texts exceeding standard attention limits[cite: 5, 6].
    """
    tokenized = tokenizer(text, truncation=False, return_attention_mask=False)
    full_ids = tokenized["input_ids"]
    
    chunks = []
    step = max_length - stride
    
    if len(full_ids) <= max_length:
        return [full_ids]
        
    for i in range(0, len(full_ids), step):
        chunk = full_ids[i : i + max_length]
        chunks.append(chunk)
        if i + max_length >= len(full_ids):
            break
            
    return chunks

def prepare_dataset_records(raw_records, tokenizer, max_length=1024, stride=256):
    """
    Optimized long-document handling: Selects key strategic segments 
    (e.g., the beginning/facts and the end/ruling) instead of exhaustive chunk explosion.
    """
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    for record in raw_records:
        doc_text = record.get("text", "")
        label = record.get("label", 0)
        
        tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
        full_ids = tokenized["input_ids"]
        
        if len(full_ids) <= max_length:
            chunks = [full_ids]
        else:
            # Take the first chunk (facts/arguments) and the last chunk (rulings/conclusion)
            # This captures the dual anchors of legal judgments efficiently.
            first_chunk = full_ids[:max_length]
            last_chunk = full_ids[-max_length:]
            chunks = [first_chunk, last_chunk]
            
        for chunk in chunks:
            input_ids_list.append(chunk)
            attention_mask_list.append([1] * len(chunk))
            labels_list.append(label)
            
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "label": labels_list
    }

print("Processing Training and Development Datasets...")
train_features = prepare_dataset_records(train_data, tokenizer)
dev_features = prepare_dataset_records(dev_data, tokenizer)

train_dataset = Dataset.from_dict(train_features)
dev_dataset = Dataset.from_dict(dev_features)


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Processing Training and Development Datasets...


In [5]:
import torch
from torch.utils.data import DataLoader

def custom_collate_fn(batch):
    # Extract fields from the list of dictionary items in the batch
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["label"], dtype=torch.long) for item in batch]
    
    # Pad sequences to the maximum length in this specific batch
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "label": labels
    }

# Re-initialize DataLoaders with the custom collate function
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=4, shuffle=False, collate_fn=custom_collate_fn)

print("DataLoaders re-initialized successfully with padding collation!")

DataLoaders re-initialized successfully with padding collation!


In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support
import zipfile
import os

# ==========================================
# 1. RMS NORM (Modern Decoder Component)
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


# ==========================================
# 2. CUSTOM LoRA LINEAR LAYER (LORMA)
# ==========================================
class LoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, r: int = 16, lora_alpha: int = 32, lora_dropout: float = 0.05):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.scaling = lora_alpha / r
        
        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        self.bias = nn.Parameter(torch.empty(out_features), requires_grad=False)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)

        if r > 0:
            self.lora_A = nn.Parameter(torch.zeros(r, in_features))
            self.lora_B = nn.Parameter(torch.zeros(out_features, r))
            self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)
        else:
            self.lora_A = None
            self.lora_B = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = F.linear(x, self.weight, self.bias)
        if self.r > 0 and self.lora_A is not None and self.lora_B is not None:
            lora_out = F.linear(self.lora_dropout(x), self.lora_A)
            lora_out = F.linear(lora_out, self.lora_B)
            return base_out + (lora_out * self.scaling)
        return base_out


# ==========================================
# 3. GROUPED-QUERY ATTENTION (GQA)
# ==========================================
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_query_heads: int, num_kv_heads: int, r: int = 16):
        super().__init__()
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        
        assert num_query_heads % num_kv_heads == 0, "Query heads must be divisible by KV heads."
        self.num_queries_per_kv = num_query_heads // num_kv_heads
        self.head_dim = d_model // num_query_heads
        
        self.q_proj = LoRALinear(d_model, d_model, r=r)
        self.k_proj = LoRALinear(d_model, num_kv_heads * self.head_dim, r=r)
        self.v_proj = LoRALinear(d_model, num_kv_heads * self.head_dim, r=r)
        self.out_proj = LoRALinear(d_model, d_model, r=r)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        
        q = self.q_proj(x).view(batch_size, seq_len, self.num_query_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-10000.0'))
            
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v)
        
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(context)


# ==========================================
# 4. SWIGLU FEED-FORWARD NETWORK (Modern FFN)
# ==========================================
class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int, r: int = 16):
        super().__init__()
        self.gate_proj = LoRALinear(d_model, hidden_dim, r=r)
        self.up_proj = LoRALinear(d_model, hidden_dim, r=r)
        self.down_proj = LoRALinear(hidden_dim, d_model, r=r)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU activation formula: swish(gate(x)) * up(x)
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


# ==========================================
# 5. MODERN DECODER BLOCK & CLASSIFIER
# ==========================================
class ModernDecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_query_heads: int, num_kv_heads: int, r: int = 16):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, num_query_heads, num_kv_heads, r=r)
        self.norm2 = RMSNorm(d_model)
        
        # Hidden dimension for SwiGLU is typically ~8/3 of d_model or 4x
        hidden_dim = int(8 * d_model / 3)
        self.ffn = SwiGLUFFN(d_model, hidden_dim, r=r)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class ModernDecoderLORMAClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_query_heads: int, num_kv_heads: int, num_labels: int, r: int = 16, max_len: int = 1024):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        
        self.blocks = nn.ModuleList([
            ModernDecoderBlock(d_model, num_query_heads, num_kv_heads, r=r) 
            for _ in range(num_layers)
        ])
        
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape
        if seq_len > self.pos_embedding.num_embeddings:
            input_ids = input_ids[:, :self.pos_embedding.num_embeddings]
            if attention_mask is not None:
                attention_mask = attention_mask[:, :self.pos_embedding.num_embeddings]
            seq_len = self.pos_embedding.num_embeddings

        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.token_embedding(input_ids) + self.pos_embedding(positions)
        
        mask = None
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
            
        for block in self.blocks:
            x = block(x, mask)
            
        x = self.norm_f(x)
        
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_embeddings = torch.sum(x * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            pooled = sum_embeddings / sum_mask
        else:
            pooled = x.mean(dim=1)
            
        logits = self.classifier(pooled)
        return logits


# ==========================================
# 6. MODEL INITIALIZATION & TRAINING EXECUTION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = len(tokenizer) if 'tokenizer' in globals() else 30522
D_MODEL = 256
NUM_LAYERS = 4
NUM_QUERY_HEADS = 8
NUM_KV_HEADS = 2
NUM_LABELS = 2
LR = 3e-4
EPOCHS = 5

print("Initializing Modern Decoder LORMA Classifier from scratch...")
model = ModernDecoderLORMAClassifier(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    num_query_heads=NUM_QUERY_HEADS,
    num_kv_heads=NUM_KV_HEADS,
    num_labels=NUM_LABELS,
    r=16
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = nn.CrossEntropyLoss()

print("\nStarting Modern Decoder LORMA Training Loop...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss / len(train_loader):.4f}")
    
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.inference_mode():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            logits = model(input_ids, attention_mask)
            preds = logits.argmax(dim=-1).cpu().tolist()
            
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())
            
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    print(f"Validation Macro-F1: {f1:.4f} (Precision: {precision:.4f}, Recall: {recall:.4f})")

print("\nModern Decoder LORMA training pipeline complete!")

Initializing Modern Decoder LORMA Classifier from scratch...

Starting Modern Decoder LORMA Training Loop...
Epoch 1/5 | Train Loss: 0.6527
Validation Macro-F1: 0.5594 (Precision: 0.5957, Recall: 0.5788)
Epoch 2/5 | Train Loss: 0.5894
Validation Macro-F1: 0.5381 (Precision: 0.6009, Recall: 0.5715)
Epoch 3/5 | Train Loss: 0.5291
Validation Macro-F1: 0.5876 (Precision: 0.5877, Recall: 0.5877)
Epoch 4/5 | Train Loss: 0.4741
Validation Macro-F1: 0.5641 (Precision: 0.5641, Recall: 0.5641)
Epoch 5/5 | Train Loss: 0.4225
Validation Macro-F1: 0.5659 (Precision: 0.5672, Recall: 0.5667)

Modern Decoder LORMA training pipeline complete!


In [7]:
# ==========================================
# 6. INFERENCE & SUBMISSION GENERATION
# ==========================================
print("Loading Test Data for Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_4_CJPE/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

model.eval()
predictions = []

for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    # Tokenize input test text
    encoded = tokenizer(
        case_text, 
        truncation=False, 
        return_attention_mask=False
    )
    full_ids = encoded["input_ids"]
    
    # Apply the same 1024 token truncation used during training
    chunk = full_ids[:1024]
    
    input_ids = torch.tensor([chunk], dtype=torch.long).to(device)
    attention_mask = torch.tensor([[1] * len(chunk)], dtype=torch.long).to(device)
    
    with torch.inference_mode():
        # Directly get logits from the scratch model forward pass
        logits = model(input_ids, attention_mask)
        pred_label = int(logits.argmax(dim=-1).item())
        
    predictions.append({
        "id": case_id,
        "pred": pred_label
    })

# Save to expected submission format (`prediction.json` zipped)
output_dir = "./phase4_gqa_lorma_cjpe"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nCJPE Prediction complete! Generated {len(predictions)} predictions.")
print(f"Submission saved to: {sub_zip_path}")

Loading Test Data for Submission Generation...


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            


CJPE Prediction complete! Generated 1517 predictions.
Submission saved to: ./phase4_gqa_lorma_cjpe/submission.zip
